# M01 · Regresión logística

Modelo lineal de referencia de la fase 2. Usamos el preprocesado y la evaluación comunes de `src/modeling.py`, para que todos los modelos se comparen igual.

**Reglas del equipo (EDA, paso 14.4):**
- Solo `train`. El test no se toca hasta M08.
- Validación cruzada con `StratifiedGroupKFold` agrupando por `grupo_producto`.
- Métrica principal: **F1 macro**, con recall por clase (sobre todo NOVA 2). Objetivo: gap train/CV < 0,05.
- Imputación y escalado dentro del pipeline, ajustados solo con el train de cada fold.

**Experimentos:**
1. Baseline: sin `log1p`.
2. Con `log1p` en `n_ingredientes`, `additives_n` y `salt_100g`.
3. Con `class_weight="balanced"` (NOVA 4 es el 55,5 %, NOVA 2 el 11,2 %).
4. Sin `energy-kcal_100g` y `saturated-fat_100g` (redundantes con los macros, paso 13).
5. Ajuste de la regularización `C`.

In [1]:
import sys
sys.path.insert(0, "..")   # para importar src/ desde notebooks/

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.modeling import (TARGET, GROUP, LABELS, load_train, get_feature_columns,
                          build_preprocessor, evaluate_cv)

In [2]:
df = load_train()
print(df.shape)
print(df[TARGET].value_counts(normalize=True).sort_index().round(3))

DROP_REDUNDANT = ["energy-kcal_100g", "saturated-fat_100g"]
y, groups = df[TARGET], df[GROUP]

(38495, 35)
nova_group
1    0.129
2    0.112
3    0.204
4    0.555
Name: proportion, dtype: float64


## Función auxiliar

Un solo sitio para construir y evaluar cada variante. Guardamos los resultados en un diccionario para compararlos al final.

In [3]:
results = {}

def run(name, log_skewed=True, class_weight=None, drop=(), C=1.0):
    cols = get_feature_columns(df, drop=drop)
    model = Pipeline([
        ("prep", build_preprocessor(cols, log_skewed=log_skewed)),
        ("clf", LogisticRegression(C=C, class_weight=class_weight, max_iter=2000)),
    ])
    res = evaluate_cv(model, df[cols["all"]], y, groups)
    results[name] = res
    print(f"{name}: F1 macro CV {res['f1_macro_cv']} (±{res['f1_macro_cv_std']}), "
          f"gap {res['gap']}, recall {res['recall']}")
    return res

## 1. Baseline (sin `log1p`)

In [4]:
run("1 baseline", log_skewed=False);

1 baseline: F1 macro CV 0.95 (±0.0015), gap 0.0015, recall {'nova_1': 0.9802, 'nova_2': 0.9889, 'nova_3': 0.9269, 'nova_4': 0.9464}


## 2. Con `log1p` en las variables asimétricas

In [5]:
run("2 log1p");

2 log1p: F1 macro CV 0.9506 (±0.0013), gap 0.0018, recall {'nova_1': 0.9784, 'nova_2': 0.9877, 'nova_3': 0.9247, 'nova_4': 0.9495}


## 3. Con `class_weight="balanced"`

Da más peso a las clases pequeñas. Esperamos que suba el recall de NOVA 1-3 y baje el de NOVA 4.

In [6]:
run("3 log1p + balanced", class_weight="balanced");

3 log1p + balanced: F1 macro CV 0.9458 (±0.002), gap 0.0019, recall {'nova_1': 0.9879, 'nova_2': 0.9889, 'nova_3': 0.9443, 'nova_4': 0.9298}


## 4. Sin las variables redundantes

In [7]:
run("4a log1p, sin redundantes", drop=DROP_REDUNDANT)
run("4b log1p + balanced, sin redundantes", class_weight="balanced", drop=DROP_REDUNDANT);

4a log1p, sin redundantes: F1 macro CV 0.9508 (±0.0009), gap 0.0014, recall {'nova_1': 0.9792, 'nova_2': 0.9872, 'nova_3': 0.9251, 'nova_4': 0.9495}
4b log1p + balanced, sin redundantes: F1 macro CV 0.9462 (±0.002), gap 0.0014, recall {'nova_1': 0.9887, 'nova_2': 0.9889, 'nova_3': 0.9446, 'nova_4': 0.9302}


## 5. Regularización

`C` es la inversa de la fuerza de la regularización L2 (más pequeño = más regularizado). Probamos varios valores con la mejor configuración de arriba.

In [8]:
for c in [0.01, 0.1, 1, 10, 100]:
    run(f"5 log1p, C={c}", C=c)

5 log1p, C=0.01: F1 macro CV 0.9205 (±0.0032), gap 0.0011, recall {'nova_1': 0.9368, 'nova_2': 0.9615, 'nova_3': 0.8648, 'nova_4': 0.9453}
5 log1p, C=0.1: F1 macro CV 0.9451 (±0.0025), gap 0.0011, recall {'nova_1': 0.9742, 'nova_2': 0.9766, 'nova_3': 0.9172, 'nova_4': 0.9489}
5 log1p, C=1: F1 macro CV 0.9506 (±0.0013), gap 0.0018, recall {'nova_1': 0.9784, 'nova_2': 0.9877, 'nova_3': 0.9247, 'nova_4': 0.9495}
5 log1p, C=10: F1 macro CV 0.95 (±0.0011), gap 0.0025, recall {'nova_1': 0.9758, 'nova_2': 0.9875, 'nova_3': 0.9235, 'nova_4': 0.9496}
5 log1p, C=100: F1 macro CV 0.9503 (±0.0009), gap 0.0021, recall {'nova_1': 0.9756, 'nova_2': 0.9877, 'nova_3': 0.925, 'nova_4': 0.9496}


## Comparación

In [9]:
table = pd.DataFrame({
    name: {"F1 macro train": r["f1_macro_train"], "F1 macro CV": r["f1_macro_cv"],
           "std": r["f1_macro_cv_std"], "gap": r["gap"],
           **{f"recall {k}": v for k, v in r["recall"].items()}}
    for name, r in results.items()
}).T
table.sort_values("F1 macro CV", ascending=False)

,F1 macro train,F1 macro CV,std,gap,recall nova_1,recall nova_2,recall nova_3,recall nova_4
"4a log1p, sin redundantes",0.9522,0.9508,0.0009,0.0014,0.9792,0.9872,0.9251,0.9495
2 log1p,0.9523,0.9506,0.0013,0.0018,0.9784,0.9877,0.9247,0.9495
"5 log1p, C=1",0.9523,0.9506,0.0013,0.0018,0.9784,0.9877,0.9247,0.9495
"5 log1p, C=100",0.9523,0.9503,0.0009,0.0021,0.9756,0.9877,0.9250,0.9496
1 baseline,0.9515,0.9500,0.0015,0.0015,0.9802,0.9889,0.9269,0.9464
"5 log1p, C=10",0.9525,0.9500,0.0011,0.0025,0.9758,0.9875,0.9235,0.9496
"4b log1p + balanced, sin redundantes",0.9476,0.9462,0.0020,0.0014,0.9887,0.9889,0.9446,0.9302
3 log1p + balanced,0.9477,0.9458,0.0020,0.0019,0.9879,0.9889,0.9443,0.9298
"5 log1p, C=0.1",0.9462,0.9451,0.0025,0.0011,0.9742,0.9766,0.9172,0.9489
"5 log1p, C=0.01",0.9216,0.9205,0.0032,0.0011,0.9368,0.9615,0.8648,0.9453


## Matriz de confusión de la mejor variante (filas = real, columnas = predicho)

In [10]:
best = table["F1 macro CV"].idxmax()
print(best)
pd.DataFrame(results[best]["confusion_matrix"],
             index=[f"real {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])

4a log1p, sin redundantes


,pred 1,pred 2,pred 3,pred 4
real 1,4849,28,40,35
real 2,21,4253,20,14
real 3,233,22,7276,334
real 4,195,10,874,20291


## Conclusiones

- **La regresión logística llega a un F1 macro de ≈ 0,95 en CV**, con un gap train/CV de ≈ 0,002 (objetivo < 0,05). No hay sobreajuste. Era lo esperado: el EDA ya avisaba de que OFF calcula NOVA con reglas parecidas a nuestras variables, así que parte de este nivel es el modelo aprendiendo el algoritmo de OFF.
- **`log1p`, quitar las variables redundantes y los valores de `C` entre 1 y 100 no cambian el resultado de forma apreciable:** las diferencias (0,950-0,951) son del orden de la desviación entre folds (0,001-0,002). Con `C` muy pequeño (0,01) el modelo se regulariza demasiado y baja a 0,92.
- **`class_weight="balanced"` mueve el equilibrio entre clases, no mejora el F1 macro:** sube el recall de NOVA 1 (0,978 → 0,988) y NOVA 3 (0,925 → 0,944), y baja el de NOVA 4 (0,950 → 0,930). El F1 macro baja ≈ 0,005.
- **NOVA 2 está resuelta** (recall ≈ 0,99, como anticipaba el EDA: grasa, `Fats` y `sin_lista_ingredientes`).
- **El error está en la frontera 3-4:** en la matriz de confusión (variante 4a), 874 NOVA 4 se predicen como 3 y 334 NOVA 3 como 4. Es el punto a mejorar con modelos no lineales (árboles, boosting).
- **También se mezclan algo NOVA 1 y 3:** 233 NOVA 3 se predicen como NOVA 1. El EDA decía que se separan por los ingredientes añadidos (sal, azúcar, aceite), y aquí la frontera no es limpia. Es un error menor que el de 3-4, pero conviene vigilarlo en los modelos siguientes.
- **Referencia para M02 en adelante:** 4a (`log1p`, sin `energy-kcal_100g` ni `saturated-fat_100g`, `C=1`, `class_weight=None`) con F1 macro CV 0,9508; es un empate técnico con la variante con todas las variables, pero tiene menos variables.
- El test no se ha usado.